In [ ]:
import json
import os
import glob
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Make the project root importable so `from src.bots import ...` works
sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
from src.artifacts_store import (
    apply_bot_cache,
    load_classifier,
    try_load_game_cache,
)

N_PREVIEW = 5

re_human_cache = try_load_game_cache("re", "human")
re_bot_cache = try_load_game_cache("re", "bot")
if re_human_cache is None or re_bot_cache is None:
    raise FileNotFoundError("re cache")
re_games_df = re_human_cache["features"]
apply_bot_cache("re", re_bot_cache, globals(), n_preview=N_PREVIEW)

lol_human_cache = try_load_game_cache("lol", "human")
lol_bot_cache = try_load_game_cache("lol", "bot")
if lol_human_cache is None or lol_bot_cache is None:
    raise FileNotFoundError("lol cache")
lol_games_df = lol_human_cache["features"]
lol_mouse_by_game = dict(zip(lol_human_cache["session_ids"], lol_human_cache["traces"]))
apply_bot_cache("lol", lol_bot_cache, globals(), n_preview=N_PREVIEW)

csgo_human_cache = try_load_game_cache("csgo", "human")
csgo_bot_cache = try_load_game_cache("csgo", "bot")
if csgo_human_cache is None or csgo_bot_cache is None:
    raise FileNotFoundError("csgo cache")
csgo_games_df = csgo_human_cache["features"]
csgo_mouse_win = {}
for sid, trace in zip(csgo_human_cache["session_ids"], csgo_human_cache["traces"]):
    session, participant = sid.split("_", 1)
    csgo_mouse_win[(session, participant)] = trace
apply_bot_cache("csgo", csgo_bot_cache, globals(), n_preview=N_PREVIEW)

for bot_type in ("stitch", "smooth", "bezier", "vae"):
    globals()[f"re_model_{bot_type}"] = load_classifier("re", "raw", bot_type)
    globals()[f"m_si_{bot_type}"] = load_classifier("re", "si_min", bot_type)
    globals()[f"m_si_ext_{bot_type}"] = load_classifier("re", "si_ext", bot_type)
    globals()[f"csgo_x_model_{bot_type}"] = load_classifier("csgo", "raw", bot_type)
    globals()[f"m_csgo_si_{bot_type}"] = load_classifier("csgo", "si_min", bot_type)
    globals()[f"m_csgo_si_ext_{bot_type}"] = load_classifier("csgo", "si_ext", bot_type)


## Feature scale comparison (RE vs LoL)

In [ ]:
from src.features import cross_game_feature_cols

cols = cross_game_feature_cols

print("Feature medians (RE human vs LoL human vs LoL bots):")
compare = pd.DataFrame({
    "RE_human": re_games_df[cols].median(),
    "LoL_human": lol_games_df[cols].median(),
    "LoL_stitch": lol_bots_stitch_df[cols].median(),
    "LoL_smooth": lol_bots_smooth_df[cols].median(),
    "LoL_bezier": lol_bots_bezier_df[cols].median(),
    "LoL_vae": lol_bots_vae_df[cols].median(),
}).round(3)
print(compare)


## RE → LoL diagnose (raw features)

In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import cross_game_feature_cols

print("=== Raw features: true zero-shot RE -> LoL ===")
print("(threshold from LoL humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", re_model_stitch, lol_games_df, lol_bots_stitch_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "smooth", re_model_smooth, lol_games_df, lol_bots_smooth_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "bezier", re_model_bezier, lol_games_df, lol_bots_bezier_df,
    cross_game_feature_cols, title_suffix="raw",
)

print()
_ = diagnose_cross_game(
    "vae", re_model_vae, lol_games_df, lol_bots_vae_df,
    cross_game_feature_cols, title_suffix="raw",
)


## RE → LoL diagnose (scale-invariant features)

In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import to_scale_invariant, SCALE_INVARIANT_COLS

lol_si_human = to_scale_invariant(lol_games_df)
lol_si_stitch = to_scale_invariant(lol_bots_stitch_df)
lol_si_smooth = to_scale_invariant(lol_bots_smooth_df)
lol_si_bezier = to_scale_invariant(lol_bots_bezier_df)
lol_si_vae = to_scale_invariant(lol_bots_vae_df)

print("=== Scale-invariant: true zero-shot RE -> LoL ===")
print("(threshold from LoL humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", m_si_stitch, lol_si_human, lol_si_stitch,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)
print()
_ = diagnose_cross_game(
    "smooth", m_si_smooth, lol_si_human, lol_si_smooth,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)
print()
_ = diagnose_cross_game(
    "bezier", m_si_bezier, lol_si_human, lol_si_bezier,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)

print()
_ = diagnose_cross_game(
    "vae", m_si_vae, lol_si_human, lol_si_vae,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)


## RE → LoL diagnose (scale-invariant EXT)


In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import SCALE_INVARIANT_EXT_COLS

print("=== Scale-invariant EXT (10 feats): true zero-shot RE -> LoL ===")
print("(threshold from LoL humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", m_si_ext_stitch, lol_si_human, lol_si_stitch,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-EXT",
)
print()
_ = diagnose_cross_game(
    "smooth", m_si_ext_smooth, lol_si_human, lol_si_smooth,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-EXT",
)
print()
_ = diagnose_cross_game(
    "bezier", m_si_ext_bezier, lol_si_human, lol_si_bezier,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-EXT",
)
print()
_ = diagnose_cross_game(
    "vae", m_si_ext_vae, lol_si_human, lol_si_vae,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-EXT",
)


## SI-EXT ablation: drop `xy_corr` / `vh_ratio` (RE → LoL)

In [ ]:
import pandas as pd
from src.evaluation import train_bot_detector, diagnose_cross_game
from src.features import to_scale_invariant, SCALE_INVARIANT_EXT_COLS
from src.config import RNG_SEED

EXT_FULL = list(SCALE_INVARIANT_EXT_COLS)
EXT_NO_XY = [c for c in EXT_FULL if c != "xy_corr"]
EXT_NO_XY_VH = [c for c in EXT_NO_XY if c != "vh_ratio"]

re_si_human = to_scale_invariant(re_games_df)
re_bots = {
    "stitch": to_scale_invariant(re_stitch_df),
    "smooth": to_scale_invariant(re_smooth_df),
    "bezier": to_scale_invariant(re_bezier_df),
    "vae": to_scale_invariant(re_vae_df),
}
lol_si_human = to_scale_invariant(lol_games_df)
lol_bots = {
    "stitch": to_scale_invariant(lol_bots_stitch_df),
    "smooth": to_scale_invariant(lol_bots_smooth_df),
    "bezier": to_scale_invariant(lol_bots_bezier_df),
    "vae": to_scale_invariant(lol_bots_vae_df),
}

ablations = {
    "EXT-10": EXT_FULL,
    "EXT-xy": EXT_NO_XY,
    "EXT-xy-vh": EXT_NO_XY_VH,
}

rows = []
for abl_name, cols in ablations.items():
    print(f"\n{'=' * 60}")
    print(f"=== Ablation {abl_name} ({len(cols)} feats): {cols} ===")
    for bot in ["stitch", "smooth", "bezier", "vae"]:
        model, _ = train_bot_detector(
            re_si_human, re_bots[bot], cols,
            random_state=RNG_SEED, name=f"{abl_name} {bot}",
            show_feature_importance=False,
        )
        d = diagnose_cross_game(
            bot, model, lol_si_human, lol_bots[bot], cols,
            title_suffix=f"RE→LoL {abl_name}",
            plot=False, plot_proba=False,
        )
        rows.append({
            "ablation": abl_name,
            "n_feats": len(cols),
            "bot": bot,
            "auc": d["auc"],
            "cal_detect": d["detect_cal"],
            "cal_fp": d["fp_cal"],
            "thr": d["thr"],
            "detect_05": d["detect_05"],
            "fp_05": d["fp_05"],
        })
        print()

summary = pd.DataFrame(rows)
print("\n=== RE→LoL SI-EXT ablation summary ===")
print(
    summary.pivot_table(
        index="bot", columns="ablation",
        values=["auc", "cal_detect"],
    ).round(3).to_string()
)
print("\n(VAE focus: does cal_detect stay high after dropping vh_ratio?)")
print(summary[summary["bot"] == "vae"][
    ["ablation", "n_feats", "auc", "cal_detect", "cal_fp", "thr"]
].to_string(index=False))


## RE→LoL Bézier inversion diagnostic


In [ ]:
import pandas as pd
from src.features import cross_game_feature_cols, to_scale_invariant, SCALE_INVARIANT_COLS

raw_cols = cross_game_feature_cols

# --- (1) Feature medians: RE human / RE bezier / LoL human / LoL bezier ---
print("=== Raw 7-feature medians ===")
raw_med = pd.DataFrame({
    "RE_human": re_games_df[raw_cols].median(),
    "RE_bezier": re_bezier_df[raw_cols].median(),
    "LoL_human": lol_games_df[raw_cols].median(),
    "LoL_bezier": lol_bots_bezier_df[raw_cols].median(),
}).round(4)
# signed gap human - bot (same game); flip if RE and LoL gaps have opposite sign
raw_med["gap_RE"] = (raw_med["RE_human"] - raw_med["RE_bezier"]).round(4)
raw_med["gap_LoL"] = (raw_med["LoL_human"] - raw_med["LoL_bezier"]).round(4)
raw_med["sign_flip"] = (raw_med["gap_RE"] * raw_med["gap_LoL"]) < 0
print(raw_med)
print()

re_sf = to_scale_invariant(re_games_df)
re_bz_sf = to_scale_invariant(re_bezier_df)
lol_sf = to_scale_invariant(lol_games_df)
lol_bz_sf = to_scale_invariant(lol_bots_bezier_df)

print("=== Scale-invariant 4-feature medians ===")
sf_med = pd.DataFrame({
    "RE_human": re_sf[SCALE_INVARIANT_COLS].median(),
    "RE_bezier": re_bz_sf[SCALE_INVARIANT_COLS].median(),
    "LoL_human": lol_sf[SCALE_INVARIANT_COLS].median(),
    "LoL_bezier": lol_bz_sf[SCALE_INVARIANT_COLS].median(),
}).round(4)
sf_med["gap_RE"] = (sf_med["RE_human"] - sf_med["RE_bezier"]).round(4)
sf_med["gap_LoL"] = (sf_med["LoL_human"] - sf_med["LoL_bezier"]).round(4)
sf_med["sign_flip"] = (sf_med["gap_RE"] * sf_med["gap_LoL"]) < 0
print(sf_med)
print()

# --- (2) Feature importance of the RE-trained bezier detectors ---
print("=== RE bezier model importance (raw 7-feat, used in RE→LoL raw) ===")
imp_raw = pd.Series(
    re_model_bezier.feature_importances_, index=raw_cols
).sort_values(ascending=False)
print(imp_raw.round(4))
print()

print("=== RE bezier model importance (scale-invariant, used in RE→LoL SF) ===")
imp_sf = pd.Series(
    m_si_bezier.feature_importances_, index=SCALE_INVARIANT_COLS
).sort_values(ascending=False)
print(imp_sf.round(4))
print()

# --- Cross-read: high importance ∩ sign flip ---
print("=== Suspects: importance rank + sign_flip ===")
print("Raw:")
for feat, imp in imp_raw.items():
    flip = bool(raw_med.loc[feat, "sign_flip"])
    print(f"  {feat:16s}  imp={imp:.3f}  flip={flip}  "
          f"gap_RE={raw_med.loc[feat, 'gap_RE']:+.4f}  gap_LoL={raw_med.loc[feat, 'gap_LoL']:+.4f}")
print("Scale-invariant:")
for feat, imp in imp_sf.items():
    flip = bool(sf_med.loc[feat, "sign_flip"])
    print(f"  {feat:16s}  imp={imp:.3f}  flip={flip}  "
          f"gap_RE={sf_med.loc[feat, 'gap_RE']:+.4f}  gap_LoL={sf_med.loc[feat, 'gap_LoL']:+.4f}")



## RE→LoL Bézier diagnostic (scale-invariant EXT medians)


In [ ]:
import pandas as pd
from src.features import to_scale_invariant, SCALE_INVARIANT_EXT_COLS

re_sf = to_scale_invariant(re_games_df)
re_bz_sf = to_scale_invariant(re_bezier_df)
lol_sf = to_scale_invariant(lol_games_df)
lol_bz_sf = to_scale_invariant(lol_bots_bezier_df)

print("=== Scale-invariant EXT 10-feature medians ===")
sf_med = pd.DataFrame({
    "RE_human": re_sf[SCALE_INVARIANT_EXT_COLS].median(),
    "RE_bezier": re_bz_sf[SCALE_INVARIANT_EXT_COLS].median(),
    "LoL_human": lol_sf[SCALE_INVARIANT_EXT_COLS].median(),
    "LoL_bezier": lol_bz_sf[SCALE_INVARIANT_EXT_COLS].median(),
}).round(4)
sf_med["gap_RE"] = (sf_med["RE_human"] - sf_med["RE_bezier"]).round(4)
sf_med["gap_LoL"] = (sf_med["LoL_human"] - sf_med["LoL_bezier"]).round(4)
sf_med["sign_flip"] = (sf_med["gap_RE"] * sf_med["gap_LoL"]) < 0
print(sf_med)
print()

print("=== RE bezier model importance (SI-EXT, used in RE→LoL EXT) ===")
imp_sf = pd.Series(
    m_si_ext_bezier.feature_importances_, index=SCALE_INVARIANT_EXT_COLS
).sort_values(ascending=False)
print(imp_sf.round(4))
print()

print("=== Suspects: SI-EXT importance + sign_flip ===")
for feat, imp in imp_sf.items():
    flip = bool(sf_med.loc[feat, "sign_flip"])
    print(f"  {feat:16s}  imp={imp:.3f}  flip={flip}  "
          f"gap_RE={sf_med.loc[feat, 'gap_RE']:+.4f}  gap_LoL={sf_med.loc[feat, 'gap_LoL']:+.4f}")


## Feature importance (scale-invariant models)



In [ ]:
import pandas as pd
from src.features import SCALE_INVARIANT_COLS, SCALE_INVARIANT_EXT_COLS

imp = pd.Series(m_si_stitch.feature_importances_, index=SCALE_INVARIANT_COLS).sort_values(ascending=False)
print("=== Feature importance (scale-invariant stitch model) ===")
print(imp)
print()
imp2 = pd.Series(m_si_smooth.feature_importances_, index=SCALE_INVARIANT_COLS).sort_values(ascending=False)
print("=== Feature importance (scale-invariant smooth model) ===")
print(imp2)
print()
imp3 = pd.Series(m_si_bezier.feature_importances_, index=SCALE_INVARIANT_COLS).sort_values(ascending=False)
print("=== Feature importance (scale-invariant bezier model) ===")
print(imp3)
print()
imp4 = pd.Series(m_si_vae.feature_importances_, index=SCALE_INVARIANT_COLS).sort_values(ascending=False)
print("=== Feature importance (scale-invariant vae model) ===")
print(imp4)

imp_ext = pd.Series(m_si_ext_stitch.feature_importances_, index=SCALE_INVARIANT_EXT_COLS).sort_values(ascending=False)
print("=== Feature importance (SI-EXT stitch) ===")
print(imp_ext)
print()
imp_ext2 = pd.Series(m_si_ext_smooth.feature_importances_, index=SCALE_INVARIANT_EXT_COLS).sort_values(ascending=False)
print("=== Feature importance (SI-EXT smooth) ===")
print(imp_ext2)
print()
imp_ext3 = pd.Series(m_si_ext_bezier.feature_importances_, index=SCALE_INVARIANT_EXT_COLS).sort_values(ascending=False)
print("=== Feature importance (SI-EXT bezier) ===")
print(imp_ext3)
print()
imp_ext4 = pd.Series(m_si_ext_vae.feature_importances_, index=SCALE_INVARIANT_EXT_COLS).sort_values(ascending=False)
print("=== Feature importance (SI-EXT vae) ===")
print(imp_ext4)

## Zero-shot diagnose (raw features, RE → CSGO)

In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import cross_game_feature_cols

print("=== Raw features: true zero-shot RE -> CSGO ===")
print("(threshold from CSGO humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", re_model_stitch, csgo_games_df, csgo_bots_stitch_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "smooth", re_model_smooth, csgo_games_df, csgo_bots_smooth_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "bezier", re_model_bezier, csgo_games_df, csgo_bots_bezier_df,
    cross_game_feature_cols, title_suffix="raw",
)

print()
_ = diagnose_cross_game(
    "vae", re_model_vae, csgo_games_df, csgo_bots_vae_df,
    cross_game_feature_cols, title_suffix="raw",
)


## Zero-shot diagnose (scale-invariant features, RE → CSGO)


In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import to_scale_invariant, SCALE_INVARIANT_COLS

csgo_si_human = to_scale_invariant(csgo_games_df)
csgo_si_stitch = to_scale_invariant(csgo_bots_stitch_df)
csgo_si_smooth = to_scale_invariant(csgo_bots_smooth_df)
csgo_si_bezier = to_scale_invariant(csgo_bots_bezier_df)
csgo_si_vae = to_scale_invariant(csgo_bots_vae_df)

print("=== Scale-invariant: true zero-shot RE -> CSGO ===")
print("(threshold from CSGO humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", m_si_stitch, csgo_si_human, csgo_si_stitch,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)
print()
_ = diagnose_cross_game(
    "smooth", m_si_smooth, csgo_si_human, csgo_si_smooth,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)
print()
_ = diagnose_cross_game(
    "bezier", m_si_bezier, csgo_si_human, csgo_si_bezier,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)

print()
_ = diagnose_cross_game(
    "vae", m_si_vae, csgo_si_human, csgo_si_vae,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)


## Zero-shot diagnose (scale-invariant EXT, RE → CSGO)


In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import to_scale_invariant, SCALE_INVARIANT_EXT_COLS

csgo_si_human = to_scale_invariant(csgo_games_df)
csgo_si_stitch = to_scale_invariant(csgo_bots_stitch_df)
csgo_si_smooth = to_scale_invariant(csgo_bots_smooth_df)
csgo_si_bezier = to_scale_invariant(csgo_bots_bezier_df)
csgo_si_vae = to_scale_invariant(csgo_bots_vae_df)

print("=== Scale-invariant EXT (10 feats): true zero-shot RE -> CSGO ===")
print("(threshold from CSGO humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", m_si_ext_stitch, csgo_si_human, csgo_si_stitch,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-EXT",
)
print()
_ = diagnose_cross_game(
    "smooth", m_si_ext_smooth, csgo_si_human, csgo_si_smooth,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-EXT",
)
print()
_ = diagnose_cross_game(
    "bezier", m_si_ext_bezier, csgo_si_human, csgo_si_bezier,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-EXT",
)
print()
_ = diagnose_cross_game(
    "vae", m_si_ext_vae, csgo_si_human, csgo_si_vae,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-EXT",
)


## SI-EXT ablation: drop `xy_corr` / `vh_ratio` (RE → CSGO)


In [ ]:
import pandas as pd
from src.evaluation import train_bot_detector, diagnose_cross_game
from src.features import to_scale_invariant, SCALE_INVARIANT_EXT_COLS
from src.config import RNG_SEED

EXT_FULL = list(SCALE_INVARIANT_EXT_COLS)
EXT_NO_XY = [c for c in EXT_FULL if c != "xy_corr"]
EXT_NO_XY_VH = [c for c in EXT_NO_XY if c != "vh_ratio"]

re_si_human = to_scale_invariant(re_games_df)
re_bots = {
    "stitch": to_scale_invariant(re_stitch_df),
    "smooth": to_scale_invariant(re_smooth_df),
    "bezier": to_scale_invariant(re_bezier_df),
    "vae": to_scale_invariant(re_vae_df),
}
csgo_si_human = to_scale_invariant(csgo_games_df)
csgo_bots = {
    "stitch": to_scale_invariant(csgo_bots_stitch_df),
    "smooth": to_scale_invariant(csgo_bots_smooth_df),
    "bezier": to_scale_invariant(csgo_bots_bezier_df),
    "vae": to_scale_invariant(csgo_bots_vae_df),
}

ablations = {
    "EXT-10": EXT_FULL,
    "EXT-xy": EXT_NO_XY,
    "EXT-xy-vh": EXT_NO_XY_VH,
}

rows = []
for abl_name, cols in ablations.items():
    print(f"\n{'=' * 60}")
    print(f"=== Ablation {abl_name} ({len(cols)} feats) RE→CSGO ===")
    for bot in ["stitch", "smooth", "bezier", "vae"]:
        model, _ = train_bot_detector(
            re_si_human, re_bots[bot], cols,
            random_state=RNG_SEED, name=f"{abl_name} {bot}",
            show_feature_importance=False,
        )
        d = diagnose_cross_game(
            bot, model, csgo_si_human, csgo_bots[bot], cols,
            title_suffix=f"RE→CSGO {abl_name}",
            plot=False, plot_proba=False,
        )
        rows.append({
            "ablation": abl_name, "n_feats": len(cols), "bot": bot,
            "auc": d["auc"], "cal_detect": d["detect_cal"],
            "cal_fp": d["fp_cal"], "thr": d["thr"],
        })
        print()

summary = pd.DataFrame(rows)
print("\n=== RE→CSGO SI-EXT ablation summary ===")
print(
    summary.pivot_table(
        index="bot", columns="ablation", values=["auc", "cal_detect"],
    ).round(3).to_string()
)
print("\nVAE focus:")
print(summary[summary["bot"] == "vae"][
    ["ablation", "n_feats", "auc", "cal_detect", "cal_fp", "thr"]
].to_string(index=False))


## Feature scale comparison (RE vs CSGO)

In [ ]:
from src.features import cross_game_feature_cols

cols = cross_game_feature_cols

print("Feature medians (RE human vs CSGO human vs CSGO bots):")
compare_re_csgo = pd.DataFrame({
    "RE_human": re_games_df[cols].median(),
    "CSGO_human": csgo_games_df[cols].median(),
    "CSGO_stitch": csgo_bots_stitch_df[cols].median(),
    "CSGO_smooth": csgo_bots_smooth_df[cols].median(),
    "CSGO_bezier": csgo_bots_bezier_df[cols].median(),
    "CSGO_vae": csgo_bots_vae_df[cols].median(),
}).round(3)
print(compare_re_csgo)



## Zero-shot diagnose (raw features, CSGO → RE)

In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import cross_game_feature_cols

print("=== Raw features: true zero-shot CSGO -> RE ===")
print("(threshold from RE humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", csgo_x_model_stitch, re_games_df, re_stitch_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "smooth", csgo_x_model_smooth, re_games_df, re_smooth_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "bezier", csgo_x_model_bezier, re_games_df, re_bezier_df,
    cross_game_feature_cols, title_suffix="raw",
)

print()
_ = diagnose_cross_game(
    "vae", csgo_x_model_vae, re_games_df, re_vae_df,
    cross_game_feature_cols, title_suffix="raw",
)


## Zero-shot diagnose (scale-invariant features, CSGO → RE)



In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import to_scale_invariant, SCALE_INVARIANT_COLS

re_si_human = to_scale_invariant(re_games_df)
re_si_stitch = to_scale_invariant(re_stitch_df)
re_si_smooth = to_scale_invariant(re_smooth_df)
re_si_bezier = to_scale_invariant(re_bezier_df)
re_si_vae = to_scale_invariant(re_vae_df)

print("=== Scale-invariant: true zero-shot CSGO -> RE ===")
print("(threshold from RE humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", m_csgo_si_stitch, re_si_human, re_si_stitch,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)
print()
_ = diagnose_cross_game(
    "smooth", m_csgo_si_smooth, re_si_human, re_si_smooth,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)
print()
_ = diagnose_cross_game(
    "bezier", m_csgo_si_bezier, re_si_human, re_si_bezier,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)

print()
_ = diagnose_cross_game(
    "vae", m_csgo_si_vae, re_si_human, re_si_vae,
    SCALE_INVARIANT_COLS, title_suffix="scale-invariant",
)


## Zero-shot diagnose (scale-invariant EXT, CSGO → RE)


In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import to_scale_invariant, SCALE_INVARIANT_EXT_COLS

re_si_human = to_scale_invariant(re_games_df)
re_si_stitch = to_scale_invariant(re_stitch_df)
re_si_smooth = to_scale_invariant(re_smooth_df)
re_si_bezier = to_scale_invariant(re_bezier_df)
re_si_vae = to_scale_invariant(re_vae_df)

print("=== Scale-invariant EXT (10 feats): true zero-shot CSGO -> RE ===")
print("(threshold from RE humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", m_csgo_si_ext_stitch, re_si_human, re_si_stitch,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-EXT",
)
print()
_ = diagnose_cross_game(
    "smooth", m_csgo_si_ext_smooth, re_si_human, re_si_smooth,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-EXT",
)
print()
_ = diagnose_cross_game(
    "bezier", m_csgo_si_ext_bezier, re_si_human, re_si_bezier,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-EXT",
)
print()
_ = diagnose_cross_game(
    "vae", m_csgo_si_ext_vae, re_si_human, re_si_vae,
    SCALE_INVARIANT_EXT_COLS, title_suffix="SI-EXT",
)


## SI-EXT ablation: drop `xy_corr` / `vh_ratio` (CSGO → RE)


In [ ]:
import pandas as pd
from src.evaluation import train_bot_detector, diagnose_cross_game
from src.features import to_scale_invariant, SCALE_INVARIANT_EXT_COLS
from src.config import RNG_SEED

EXT_FULL = list(SCALE_INVARIANT_EXT_COLS)
EXT_NO_XY = [c for c in EXT_FULL if c != "xy_corr"]
EXT_NO_XY_VH = [c for c in EXT_NO_XY if c != "vh_ratio"]

csgo_si_human_tr = to_scale_invariant(csgo_games_df)
csgo_bots_tr = {
    "stitch": to_scale_invariant(csgo_bots_stitch_df),
    "smooth": to_scale_invariant(csgo_bots_smooth_df),
    "bezier": to_scale_invariant(csgo_bots_bezier_df),
    "vae": to_scale_invariant(csgo_bots_vae_df),
}
re_si_human = to_scale_invariant(re_games_df)
re_bots = {
    "stitch": to_scale_invariant(re_stitch_df),
    "smooth": to_scale_invariant(re_smooth_df),
    "bezier": to_scale_invariant(re_bezier_df),
    "vae": to_scale_invariant(re_vae_df),
}

ablations = {
    "EXT-10": EXT_FULL,
    "EXT-xy": EXT_NO_XY,
    "EXT-xy-vh": EXT_NO_XY_VH,
}

rows = []
for abl_name, cols in ablations.items():
    print(f"\n{'=' * 60}")
    print(f"=== Ablation {abl_name} ({len(cols)} feats) CSGO→RE ===")
    for bot in ["stitch", "smooth", "bezier", "vae"]:
        model, _ = train_bot_detector(
            csgo_si_human_tr, csgo_bots_tr[bot], cols,
            random_state=RNG_SEED, name=f"CSGO {abl_name} {bot}",
            show_feature_importance=False,
        )
        d = diagnose_cross_game(
            bot, model, re_si_human, re_bots[bot], cols,
            title_suffix=f"CSGO→RE {abl_name}",
            plot=False, plot_proba=False,
        )
        rows.append({
            "ablation": abl_name, "n_feats": len(cols), "bot": bot,
            "auc": d["auc"], "cal_detect": d["detect_cal"],
            "cal_fp": d["fp_cal"], "thr": d["thr"],
        })
        print()

summary = pd.DataFrame(rows)
print("\n=== CSGO→RE SI-EXT ablation summary ===")
print(
    summary.pivot_table(
        index="bot", columns="ablation", values=["auc", "cal_detect"],
    ).round(3).to_string()
)
print("\nVAE focus:")
print(summary[summary["bot"] == "vae"][
    ["ablation", "n_feats", "auc", "cal_detect", "cal_fp", "thr"]
].to_string(index=False))


## Feature scale comparison (CSGO vs RE)

In [ ]:
from src.features import cross_game_feature_cols

cols = cross_game_feature_cols

print("Feature medians (CSGO human vs RE human vs RE bots):")
compare_csgo_re = pd.DataFrame({
    "CSGO_human": csgo_games_df[cols].median(),
    "RE_human": re_games_df[cols].median(),
    "RE_stitch": re_stitch_df[cols].median(),
    "RE_smooth": re_smooth_df[cols].median(),
    "RE_bezier": re_bezier_df[cols].median(),
    "RE_vae": re_vae_df[cols].median(),
}).round(3)
print(compare_csgo_re)

